In [3]:
import cv2
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision

# تحميل مرة وحدة قبل التشغيل:
# curl -o face_landmarker.task https://storage.googleapis.com/
#   mediapipe-models/face_landmarker/face_landmarker/float16/1/
#   face_landmarker.task

# إعداد MediaPipe (Tasks API — أحدث إصدار)
base_options = mp_python.BaseOptions(
    model_asset_path='face_landmarker.task')
options = mp_vision.FaceLandmarkerOptions(
    base_options=base_options,
    running_mode=mp_vision.RunningMode.VIDEO,
    num_faces=1,
    min_face_detection_confidence=0.5,
    min_tracking_confidence=0.5
)
landmarker = mp_vision.FaceLandmarker.create_from_options(options)

# نقاط مهمة
NOSE = 1
LEFT_FACE = 234
RIGHT_FACE = 454
LEFT_IRIS = 468
RIGHT_IRIS = 473
LEFT_EYE_CORNERS = (33, 133)
RIGHT_EYE_CORNERS = (362, 263)

# YOLO بنضيفها بالمحاضرة الجاية


In [5]:
# ملاحظة: الـ API الجديد بيرجع لستة مباشرة — face[i]
# بدل face.landmark[i] بالنسخة القديمة
def eye_gaze_ratio(face, eye_corners, iris_point):
    left_x = face[eye_corners[0]].x
    right_x = face[eye_corners[1]].x
    iris_x = face[iris_point].x
    eye_width = right_x - left_x
    if eye_width == 0:
        return 0.5
    return (iris_x - left_x) / eye_width

def get_head_pose(face, w):
    nose_x = face[NOSE].x * w
    center_x = ((face[LEFT_FACE].x +
                  face[RIGHT_FACE].x) / 2) * w
    face_width = (face[RIGHT_FACE].x -
                  face[LEFT_FACE].x) * w
    threshold = face_width * 0.15
    offset = nose_x - center_x
    if offset < -threshold:
        return "TURNED LEFT"
    elif offset > threshold:
        return "TURNED RIGHT"
    return "FORWARD"

def get_gaze(face):
    left_ratio = eye_gaze_ratio(face, LEFT_EYE_CORNERS, LEFT_IRIS)
    right_ratio = eye_gaze_ratio(face, RIGHT_EYE_CORNERS, RIGHT_IRIS)
    avg = (left_ratio + right_ratio) / 2
    if avg < 0.40:
        return "RIGHT"
    elif avg > 0.60:
        return "LEFT"
    return "CENTER"


In [ ]:
from datetime import datetime
from pathlib import Path
import time
from ultralytics import YOLO

OBJECT_CLASSES = {
    67: "CELL PHONE",
    73: "BOOK",
    74: "CLOCK",
}
SCREENSHOT_DIR = Path("screenshots")
REPORT_DIR = Path("session_reports")
SCREENSHOT_DIR.mkdir(exist_ok=True)
REPORT_DIR.mkdir(exist_ok=True)

model = YOLO("yolov8n.pt")
landmarker = mp_vision.FaceLandmarker.create_from_options(options)
cap = cv2.VideoCapture(0)
frame_idx = 0
session_start = time.monotonic()
suspicious_started_at = None
previous_suspicious = False
previous_phone = False
previous_book = False
previous_clock = False
previous_head_alert = False
previous_gaze_alert = False
suspicious_events = 0
phone_detections = 0
book_detections = 0
clock_detections = 0
head_turn_alerts = 0
gaze_alerts = 0

try:
    while True:
        ret, frame = cap.read()
        if not ret:
            print("Unable to read from the camera.")
            break

        frame = cv2.flip(frame, 1)
        h, w, _ = frame.shape
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        frame_idx += 1
        ts_ms = frame_idx * 33

        result = landmarker.detect_for_video(mp_image, ts_ms)
        head_status = "NO FACE"
        gaze_status = "NO FACE"
        if result.face_landmarks:
            face = result.face_landmarks[0]
            head_status = get_head_pose(face, w)
            gaze_status = get_gaze(face)

        yolo_results = model(frame, classes=list(OBJECT_CLASSES), verbose=False)
        detected_objects = set()
        for detection_result in yolo_results:
            for box in detection_result.boxes:
                class_id = int(box.cls[0])
                if class_id not in OBJECT_CLASSES:
                    continue
                detected_objects.add(OBJECT_CLASSES[class_id])
                x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
                confidence = float(box.conf[0])
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 165, 255), 2)
                cv2.putText(
                    frame,
                    f"{OBJECT_CLASSES[class_id]} {confidence:.2f}",
                    (x1, max(y1 - 10, 20)),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    (0, 165, 255),
                    2,
                )

        phone_detected = "CELL PHONE" in detected_objects
        book_detected = "BOOK" in detected_objects
        clock_detected = "CLOCK" in detected_objects
        head_alert = head_status in ["TURNED LEFT", "TURNED RIGHT"]
        gaze_alert = gaze_status in ["LEFT", "RIGHT"]
        suspicious = head_alert or gaze_alert or phone_detected or book_detected or clock_detected

        if suspicious and not previous_suspicious:
            suspicious_events += 1
            screenshot_name = datetime.now().strftime("suspicious_%Y%m%d_%H%M%S_%f.jpg")
            cv2.imwrite(str(SCREENSHOT_DIR / screenshot_name), frame)
        if phone_detected and not previous_phone:
            phone_detections += 1
        if book_detected and not previous_book:
            book_detections += 1
        if clock_detected and not previous_clock:
            clock_detections += 1
        if head_alert and not previous_head_alert:
            head_turn_alerts += 1
        if gaze_alert and not previous_gaze_alert:
            gaze_alerts += 1

        if suspicious:
            if suspicious_started_at is None:
                suspicious_started_at = time.monotonic()
            suspicious_duration = time.monotonic() - suspicious_started_at
        else:
            suspicious_started_at = None
            suspicious_duration = 0

        high_alert = suspicious_duration > 3
        color = (0, 0, 255) if suspicious else (0, 255, 0)
        status = "HIGH ALERT" if high_alert else "SUSPICIOUS" if suspicious else "NORMAL"
        object_text = ", ".join(sorted(detected_objects)) if detected_objects else "NONE"

        cv2.putText(frame, f"Head: {head_status}", (20, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.65, color, 2)
        cv2.putText(frame, f"Gaze: {gaze_status}", (20, 65), cv2.FONT_HERSHEY_SIMPLEX, 0.65, color, 2)
        cv2.putText(frame, f"Objects: {object_text}", (20, 95), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
        cv2.putText(frame, f"Events: {suspicious_events}", (20, 125), cv2.FONT_HERSHEY_SIMPLEX, 0.65, color, 2)
        cv2.putText(frame, status, (20, 165), cv2.FONT_HERSHEY_SIMPLEX, 1, color, 3)

        cv2.imshow("Cheat Detection System", frame)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

finally:
    cap.release()
    cv2.destroyAllWindows()
    landmarker.close()
    session_duration = time.monotonic() - session_start
    report_time = datetime.now().strftime("%Y%m%d_%H%M%S")
    report = (
        "FINAL SESSION REPORT\n"
        "====================\n"
        f"Total Suspicious Events: {suspicious_events}\n"
        f"Phone Detections: {phone_detections}\n"
        f"Book Detections: {book_detections}\n"
        f"Clock Detections: {clock_detections}\n"
        f"Head Turn Alerts: {head_turn_alerts}\n"
        f"Gaze Alerts: {gaze_alerts}\n"
        f"Total Session Duration: {session_duration:.2f} seconds\n"
    )
    print("\n" + report)
    report_path = REPORT_DIR / f"session_report_{report_time}.txt"
    report_path.write_text(report, encoding="utf-8")
    print(f"Report saved to: {report_path}")

    previous_suspicious = suspicious
    previous_phone = phone_detected
    previous_book = book_detected
    previous_clock = clock_detected
    previous_head_alert = head_alert
    previous_gaze_alert = gaze_alert